In [2]:
import torch
import pickle
import numpy as np

In [3]:
with open("C:\\Users\\Neela\\Documents\\GitHub\\EntityAspectLinking\\Experiment_trainsmall\\picklefiles\\eal_trainsmall.pkl", 'rb') as eal:
    data = pickle.load(eal)

ent = [data[i][0] for i in range(len(data))]

asp = [data[i][1] for i in range(len(data))]

In [4]:
PTH = r"C:\Users\Neela\Documents\Github\EntityAspectLinking\Experiment_trainsmall\picklefiles"
def read_tensor(filename):
    with open(f'{PTH}\\{filename}', 'rb') as f:
        emb = np.load(f, allow_pickle = True)
    return emb

In [5]:
ent_emb = read_tensor('targetentemb_trainsmall.pkl').cpu()
asp_emb = read_tensor('aspentemb_trainsmall.pkl').cpu()
context_emb = read_tensor('contextemb_trainsmall.pkl').cpu()

In [19]:
aspect = [el[1] for el in data]
lngth = [len(el['candidate_aspects']) for el in aspect]
asp_count = sum([len(el['candidate_aspects']) for el in aspect])

In [20]:
#Case where true aspect is not in the candidate aspect as it is

i = 0
j = 0
check = 0
yo = 0
unid_ls = []
while(i < asp_count):
    aspect = asp[j]['true_aspect']
    i+=1    
    this = 0
    
    #if len(asp[j]['candidate_aspects']) == 0:
        #print("Ache", j)
    for cand in asp[j]['candidate_aspects']:
        if i >= asp_count:
            break
        if cand['aspect_name'] == aspect:
            check += 1
        if cand['aspect_name'] != aspect:
            this += 1
            casp = cand['aspect_name']
            i+=1
    if this == len(asp[j]['candidate_aspects']):
        yo += 1  
        unid_ls.append(j)  
            
    j += 1

In [21]:
#Case where true aspect is not in the candidate aspect as it is and i is stored in the list

i = 0
j = 0
ind = []
while(i<asp_count):
    aspect = asp[j]['true_aspect']
    i = i+1
    this = 0
    for cand in asp[j]["candidate_aspects"]:
        if i >= asp_count:
            break
        if j in unid_ls and aspect in str(cand["aspect_name"]):
            ind.append(i)
            i = i+1
            continue
        i = i+1
    j = j+1

In [22]:
unid_ls

[3538, 3812, 4460, 4461]

In [23]:
len(asp[4460]['candidate_aspects'])
#'C208; 1997–2003)' in 'First generation (W208/C208; 1997–2003)'

4

In [26]:
#Building the dataset for entity and asepct and context

dataset = np.zeros((asp_count, 201 + context_emb.shape[1]))
i = 0
k = 0
for _ in range(len(dataset)):
    while(k < len(lngth)):
        aspname = asp[k]['true_aspect']
        dataset[i, : ent_emb.shape[1]] = ent_emb[k]
        dataset[i, ent_emb.shape[1] : ent_emb.shape[1] + context_emb.shape[1]] = context_emb[k]
        dataset[i, ent_emb.shape[1] + context_emb.shape[1] : -1] = asp_emb[i]
        dataset[i, -1] = 1
        i += 1
        for p in range(lngth[k]):
            if k in unid_ls and aspname in asp[k]['candidate_aspects'][p]['aspect_name']:
                continue
            if asp[k]['true_aspect'] != asp[k]['candidate_aspects'][p]['aspect_name']:
                dataset[i, : ent_emb.shape[1]] = ent_emb[k]
                dataset[i, ent_emb.shape[1] : ent_emb.shape[1] + context_emb.shape[1]] = context_emb[k]
                dataset[i, ent_emb.shape[1] + context_emb.shape[1] : -1] = asp_emb[i]
                dataset[i, -1] = 0
                i += 1
        k += 1

In [27]:
with open(f'{PTH}\\baselinedataset_trainsmall.pkl', 'wb') as f:
    pickle.dump(dataset, f)
    print("Dumped")
f.close()

Dumped
